# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_raw.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_raw.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.006100,0.001768,0.992131,0.002865,0.000664,0.996471
1,0.996506,0.000362,0.003132,0.997038,0.000276,0.002686
2,0.003045,0.000595,0.996360,0.003027,0.000586,0.996388
3,0.004197,0.002691,0.993113,0.004076,0.002232,0.993692
4,0.993760,0.000033,0.006206,0.993914,0.000017,0.006069


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.011001,0.003238,0.985761,0.011256,0.003791,0.984953
1,0.475895,0.002462,0.521643,0.427169,0.002915,0.569917
2,0.998080,0.001354,0.000566,0.999067,0.000780,0.000153
3,0.986703,0.000025,0.013272,0.987436,0.000011,0.012554
4,0.007872,0.001685,0.990443,0.009200,0.002601,0.988199


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [45]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=510, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-13 11:43:27,075] A new study created in memory with name: no-name-d3e0224f-2c6d-49a4-8030-0c1bb560942f
Best trial: 8. Best value: 0.947757:   2%|███▏                                                                                                                                    | 12/510 [00:00<00:13, 36.13it/s]

[I 2026-07-13 11:43:27,219] Trial 1 finished with value: 0.9430452914402774 and parameters: {'weight_class_0': 0.0016075222143598087, 'weight_class_1': 0.0253848319773283, 'weight_class_2': 0.005627120344500736}. Best is trial 1 with value: 0.9430452914402774.
[I 2026-07-13 11:43:27,228] Trial 0 finished with value: 0.5907115112485416 and parameters: {'weight_class_0': 0.0016742285327395947, 'weight_class_1': 0.01444149831283666, 'weight_class_2': 7.13747262649839}. Best is trial 1 with value: 0.9430452914402774.
[I 2026-07-13 11:43:27,233] Trial 6 finished with value: 0.7612862127240101 and parameters: {'weight_class_0': 3.041872028361448, 'weight_class_1': 0.10912677266983431, 'weight_class_2': 0.019875026729748644}. Best is trial 1 with value: 0.9430452914402774.
[I 2026-07-13 11:43:27,239] Trial 2 finished with value: 0.8848936602831922 and parameters: {'weight_class_0': 0.961924674874167, 'weight_class_1': 6.020594849351361, 'weight_class_2': 0.08156086693627719}. Best is trial 1 

Best trial: 8. Best value: 0.947757:   5%|██████▏                                                                                                                                 | 23/510 [00:00<00:10, 46.28it/s]

[I 2026-07-13 11:43:27,390] Trial 13 finished with value: 0.7983750863689852 and parameters: {'weight_class_0': 0.054377427830902304, 'weight_class_1': 0.0015840907955032605, 'weight_class_2': 0.0010328498873944188}. Best is trial 8 with value: 0.9477571089202458.
[I 2026-07-13 11:43:27,415] Trial 14 finished with value: 0.806874640290875 and parameters: {'weight_class_0': 0.03776543819646967, 'weight_class_1': 0.0017031659955336579, 'weight_class_2': 0.0010379722130668366}. Best is trial 8 with value: 0.9477571089202458.
[I 2026-07-13 11:43:27,435] Trial 15 finished with value: 0.8092279245494115 and parameters: {'weight_class_0': 0.02705846236845676, 'weight_class_1': 0.0011107662509510507, 'weight_class_2': 0.0013621351254984797}. Best is trial 8 with value: 0.9477571089202458.
[I 2026-07-13 11:43:27,449] Trial 16 finished with value: 0.8083498579949407 and parameters: {'weight_class_0': 0.03249710280090409, 'weight_class_1': 0.0014155591135849605, 'weight_class_2': 0.00122438447836

Best trial: 8. Best value: 0.947757:   7%|█████████▎                                                                                                                              | 35/510 [00:00<00:09, 50.98it/s]

...

[I 2026-07-13 11:43:36,173] Trial 483 finished with value: 0.9495123269328819 and parameters: {'weight_class_0': 0.00815852253252738, 'weight_class_1': 0.1447738756416158, 'weight_class_2': 0.11522188399327078}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,216] Trial 484 finished with value: 0.9493212269211063 and parameters: {'weight_class_0': 0.006312207882246911, 'weight_class_1': 0.07199038070989322, 'weight_class_2': 0.07960784572283136}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,253] Trial 485 finished with value: 0.9494572782431465 and parameters: {'weight_class_0': 0.006151439706506706, 'weight_class_1': 0.09001340533319895, 'weight_class_2': 0.07152311205084502}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,261] Trial 486 finished with value: 0.9492926054756072 and parameters: {'weight_class_0': 0.005954421924720634, 'weight_class_1': 0.07148386172229176, 'weight_class_2': 0.0773431332026227

Best trial: 277. Best value: 0.949562:  99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 506/510 [00:09<00:00, 50.36it/s]

[I 2026-07-13 11:43:36,412] Trial 493 finished with value: 0.9494255533802344 and parameters: {'weight_class_0': 0.006063916424036061, 'weight_class_1': 0.07800964457564957, 'weight_class_2': 0.08094735995835844}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,434] Trial 494 finished with value: 0.9495503003068029 and parameters: {'weight_class_0': 0.005625935263460346, 'weight_class_1': 0.0941895829540638, 'weight_class_2': 0.078989084692306}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,468] Trial 496 finished with value: 0.9493342918388811 and parameters: {'weight_class_0': 0.004534381459873493, 'weight_class_1': 0.08435647310445883, 'weight_class_2': 0.08526441475185628}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,483] Trial 497 finished with value: 0.949535092256669 and parameters: {'weight_class_0': 0.006169189431739487, 'weight_class_1': 0.1031705947380858, 'weight_class_2': 0.08382470264739536}.

Best trial: 277. Best value: 0.949562: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 510/510 [00:09<00:00, 53.02it/s]

[I 2026-07-13 11:43:36,675] Trial 506 finished with value: 0.9493901406056843 and parameters: {'weight_class_0': 0.0051628136724108905, 'weight_class_1': 0.099715723785754, 'weight_class_2': 0.05718245227751937}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,679] Trial 508 finished with value: 0.9491651582581797 and parameters: {'weight_class_0': 0.0035960059933299203, 'weight_class_1': 0.04111301405354791, 'weight_class_2': 0.05803865184875706}. Best is trial 277 with value: 0.9495624141169466.
[I 2026-07-13 11:43:36,694] Trial 509 finished with value: 0.9493408133508288 and parameters: {'weight_class_0': 0.004266641118092803, 'weight_class_1': 0.061202820563150635, 'weight_class_2': 0.07176064010297077}. Best is trial 277 with value: 0.9495624141169466.

Best trial score:
0.9495624141169466

Best params:
{'weight_class_0': 0.010880207806710639, 'weight_class_1': 0.173906383023136, 'weight_class_2': 0.16713098563667145}


In [46]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [47]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [48]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.0_xgboost_submission.csv', index=False)

In [49]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
